# Set-Up

## Imports & Param Defs

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import polars as pl
from pathlib import Path
from pprint import pprint
from sklearn.impute import SimpleImputer

path = Path().resolve().parent.parent / "data"

In [12]:
# list for tracking df lengths after transformations
tmp_lens = []

## Helper Functions

In [17]:
def track_len(df: pl.DataFrame | pl.LazyFrame, tmp_lens: list):
    if isinstance(df, pl.LazyFrame):
        length = df.select(pl.len()).collect().item()
    else:
        length = df.shape[0]
    print(f"Current length:  {length:,}")
    print(f"Change in Rows:  {length - tmp_lens[-1] if tmp_lens else length:,}")
    print(f"Pct of Original: {length / tmp_lens[0] * 100 if tmp_lens else 100:.2f}%")
    tmp_lens.append(length)

# Define Data

In [18]:
master_df = pl.scan_parquet(path / "outputs/master_dataset.parquet")
track_len(master_df, tmp_lens)

Current length:  448,972
Change in Rows:  168
Pct of Original: 100.00%


In [19]:
master_df.head().show()

subject_id,hadm_id,stay_id,intime,outtime,anchor_age,gender,anchor_year,dod,admittime,dischtime,deathtime,ethnicity,edregtime,edouttime,insurance,in_year,age,outcome_inhospital_mortality,ed_los,intime_icu,time_to_icu_transfer,outcome_icu_transfer_12h,outcome_hospitalization,outcome_critical,ethnicity_was_null,n_ed_30d,n_ed_90d,n_ed_365d,next_ed_visit_time,next_ed_visit_time_diff,outcome_ed_revisit_3d,n_hosp_30d,n_icu_30d,n_hosp_90d,n_icu_90d,n_hosp_365d,…,eci_PHTN,eci_PVD,eci_HTN1,eci_HTN2,eci_Paralysis,eci_NeuroOther,eci_Pulmonary,eci_DM1,eci_DM2,eci_Hypothyroid,eci_Renal,eci_Liver,eci_PUD,eci_HIV,eci_Lymphoma,eci_Tumor2,eci_Tumor1,eci_Rheumatic,eci_Coagulopathy,eci_Obesity,eci_WeightLoss,eci_FluidsLytes,eci_BloodLoss,eci_Anemia,eci_Alcohol,eci_Drugs,eci_Psychoses,eci_Depression,ed_temperature_last,ed_heartrate_last,ed_resprate_last,ed_o2sat_last,ed_sbp_last,ed_dbp_last,ed_pain_last,n_med,n_medrecon
i64,i64,i64,datetime[μs],datetime[μs],i64,str,i64,datetime[μs],datetime[μs],datetime[μs],datetime[μs],str,str,str,str,i32,i64,bool,duration[μs],datetime[μs],duration[μs],bool,bool,bool,bool,i32,i32,i32,datetime[μs],duration[μs],bool,i32,i32,i32,i32,i32,…,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,f64,f64,f64,f64,i64,i64,f64,u32,u32
19416727,21289895,35425579,2189-06-16 21:35:00,2189-06-16 22:15:39,69,"""M""",2189,null,null,null,null,"""UNKNOWN""",null,null,null,2189,69,false,40m 39s,null,null,false,true,false,true,0,0,0,null,null,false,0,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,37.2,122.0,28.0,100.0,108,89,NaN,8,0
10628682,22653302,35851515,2133-05-12 23:17:00,2133-05-13 04:14:00,38,"""M""",2133,null,null,null,null,"""UNKNOWN""",null,null,null,2133,38,false,4h 57m,null,null,false,true,false,true,0,0,0,null,null,false,0,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,96.1,85.0,18.0,100.0,133,60,NaN,7,11
14125406,27740734,32795976,2181-11-18 18:42:00,2181-11-18 22:55:00,76,"""F""",2181,null,null,null,null,"""UNKNOWN""",null,null,null,2181,76,false,4h 13m,null,null,false,true,false,true,0,0,0,null,null,false,0,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,96.6,92.0,16.0,96.0,136,82,4.0,4,4
11665270,26049594,33012761,2162-09-09 21:24:00,2162-09-10 00:58:00,76,"""F""",2162,null,null,null,null,"""UNKNOWN""",null,null,null,2162,76,false,3h 34m,null,null,false,true,false,true,0,0,0,null,null,false,0,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,97.0,105.0,18.0,97.0,124,70,0.0,4,14
14143633,29781340,30136351,2169-10-02 05:47:00,2169-10-02 18:23:00,34,"""F""",2169,null,null,null,null,"""UNKNOWN""",null,null,null,2169,34,false,12h 36m,null,null,false,true,false,true,0,0,0,null,null,false,0,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,98.3,55.0,18.0,99.0,136,84,0.0,4,1


# 1. General Filter - Age, triage_acuity

In [ ]:
# filter out children
master_df = master_df.filter(pl.col('age') >= 18)

track_len(master_df, tmp_lens)

Current length:  448,804
Change in Rows:  -168
Pct of Original: 99.96%


In [21]:
# drop null `triage_acuity`
master_df = master_df.filter(pl.col('triage_acuity').is_not_null())

track_len(master_df, tmp_lens)

Current length:  441,437
Change in Rows:  -7,367
Pct of Original: 98.32%


# 2. Outlier Detection